# Feature Engineering
This notebook handles feature engineering for the pain medication effectiveness prediction model, including creating target variables, encoding categorical features, and extracting text features.

## 1. Import Required Libraries

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 2. Load Cleaned Data

In [2]:
# TODO: Load cleaned data
cleaned_data_path = '../data/cleaned/pain_meds_cleaned.csv'

df = pd.read_csv(cleaned_data_path)

print(f"Cleaned data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
display(df.head())

Cleaned data loaded successfully!
Shape: (2473, 8)

Columns: ['uniqueID', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'year']


,uniqueID,drugName,condition,review,rating,date,usefulCount,year
0,189138,Oxycodone,chronic pain,"""I&#039;ve been taking oxycodone for roughly 5 years....I&#039;ve gone a few weekends without it...",8,18-Dec-16,24,2016
1,80410,Aleve,back pain,"""I love Aleve! It makes all my lower back pain disappear, I feel like a new person.""",10,12-Aug-10,41,2010
2,171980,Meloxicam,osteoarthritis,"""I have been using Mobic to relieve the pain from my Spinal Fusion I had in March/2001. I was pr...",10,25-Sep-09,26,2009
3,40937,Acetaminophen / oxycodone,chronic pain,"""This med helps to take the edge off enough for the pain to be tolerable. I take HCL 15 mg one e...",6,10-Nov-17,0,2017
4,113576,Acetaminophen / butalbital / caffeine,headache,"""I have been suffering from terrible allergies due to hay fever. The allergies caused horrible ...",9,19-Sep-11,0,2011


## 3. Create Effectiveness Label (Target Variable)

In [3]:
# TODO: Create function to categorize effectiveness based on rating
# Rating 8-10: Effective
# Rating 5-7: Partially Effective
# Rating 1-4: Not Effective

def create_effectiveness_label(rating):
    """
    Categorizes medication effectiveness based on rating.
    
    Args:
        rating: Numeric rating (1-10)
    
    Returns:
        Effectiveness category (Effective, Partially Effective, Not Effective)
    """
    if rating >= 8:
        return 'Effective'
    elif rating >= 5:
        return 'Partially Effective'
    else:
        return 'Not Effective'

# Apply function to create target variable
if 'rating' in df.columns:
    df['effectiveness'] = df['rating'].apply(create_effectiveness_label)
    
    print("Effectiveness label created!")
    print("\nEffectiveness Distribution:")
    print(df['effectiveness'].value_counts())
    print("\nPercentage Distribution:")
    print(df['effectiveness'].value_counts(normalize=True) * 100)
else:
    print("Warning: 'rating' column not found!")

Effectiveness label created!

Effectiveness Distribution:
effectiveness
Effective              1788
Not Effective           410
Partially Effective     275
Name: count, dtype: int64

Percentage Distribution:
effectiveness
Effective              72.300849
Not Effective          16.579054
Partially Effective    11.120097
Name: proportion, dtype: float64


## 4. Apply Function to Create Target Variable

In [4]:
# TODO: Create numeric encoding for target variable (for classification)
# Map effectiveness labels to numeric values
effectiveness_mapping = {
    'Not Effective': 0,
    'Partially Effective': 1,
    'Effective': 2
}

if 'effectiveness' in df.columns:
    df['effectiveness_encoded'] = df['effectiveness'].map(effectiveness_mapping)
    
    print("Target variable encoding complete!")
    print("\nEncoded Distribution:")
    print(df['effectiveness_encoded'].value_counts().sort_index())
    
    # Verify mapping
    print("\nSample mapping:")
    display(df[['rating', 'effectiveness', 'effectiveness_encoded']].sample(10))

Target variable encoding complete!

Encoded Distribution:
effectiveness_encoded
0     410
1     275
2    1788
Name: count, dtype: int64

Sample mapping:


,rating,effectiveness,effectiveness_encoded
2168,10,Effective,2
130,9,Effective,2
1333,9,Effective,2
1711,9,Effective,2
10,9,Effective,2
1449,5,Partially Effective,1
240,9,Effective,2
1378,10,Effective,2
2125,10,Effective,2
1563,10,Effective,2


## 5. One-Hot Encoding for Categorical Variables

In [5]:
# TODO: Apply one-hot encoding to drugName and condition
# Limit to top N categories to avoid too many features

print("Applying one-hot encoding...")

# Get top 30 drugs and conditions to reduce dimensionality
top_drugs = df['drugName'].value_counts().head(30).index
top_conditions = df['condition'].value_counts().head(20).index

# Create binary columns for top drugs
df['drugName_top'] = df['drugName'].apply(lambda x: x if x in top_drugs else 'Other')
drug_dummies = pd.get_dummies(df['drugName_top'], prefix='drug')

# Create binary columns for top conditions
df['condition_top'] = df['condition'].apply(lambda x: x if x in top_conditions else 'other')
condition_dummies = pd.get_dummies(df['condition_top'], prefix='condition')

print(f"\nDrug features created: {drug_dummies.shape[1]}")
print(f"Condition features created: {condition_dummies.shape[1]}")

# Combine with original dataframe
df_encoded = pd.concat([df, drug_dummies, condition_dummies], axis=1)

print(f"\nNew dataset shape: {df_encoded.shape}")

Applying one-hot encoding...

Drug features created: 31
Condition features created: 13

New dataset shape: (2473, 56)


## 6. Extract Text Features from Reviews

In [6]:
# TODO: Extract text-based features from review column if available

if 'review' in df_encoded.columns:
    print("Extracting text features from reviews...")
    
    # Feature 1: Review length (character count)
    df_encoded['review_length'] = df_encoded['review'].astype(str).apply(len)
    
    # Feature 2: Word count
    df_encoded['review_word_count'] = df_encoded['review'].astype(str).apply(lambda x: len(x.split()))
    
    # Feature 3: Presence of positive keywords
    positive_keywords = ['effective', 'works', 'great', 'excellent', 'relief', 'helped', 'better']
    df_encoded['has_positive_keywords'] = df_encoded['review'].astype(str).str.lower().apply(
        lambda x: any(keyword in x for keyword in positive_keywords)
    ).astype(int)
    
    # Feature 4: Presence of negative keywords
    negative_keywords = ['ineffective', 'useless', 'terrible', 'worse', 'pain', 'side effects', 'stopped']
    df_encoded['has_negative_keywords'] = df_encoded['review'].astype(str).str.lower().apply(
        lambda x: any(keyword in x for keyword in negative_keywords)
    ).astype(int)
    
    # Feature 5: Average word length
    df_encoded['avg_word_length'] = df_encoded['review'].astype(str).apply(
        lambda x: np.mean([len(word) for word in x.split()]) if len(x.split()) > 0 else 0
    )
    
    print("Text features extracted!")
    print("\nText Feature Summary:")
    text_features = ['review_length', 'review_word_count', 'has_positive_keywords', 
                     'has_negative_keywords', 'avg_word_length']
    display(df_encoded[text_features].describe())
else:
    print("No review column found. Skipping text feature extraction.")

Extracting text features from reviews...
Text features extracted!

Text Feature Summary:


,review_length,review_word_count,has_positive_keywords,has_negative_keywords,avg_word_length
count,2473.000000,2473.000000,2473.000000,2473.000000,2473.000000
mean,336.460574,61.693894,0.513546,0.660736,4.598614
std,237.888138,43.855655,0.499918,0.473555,0.707307
min,4.000000,1.000000,0.000000,0.000000,3.266667
25%,145.000000,26.000000,0.000000,0.000000,4.198630
50%,291.000000,53.000000,1.000000,1.000000,4.478261
75%,503.000000,92.000000,1.000000,1.000000,4.821429
max,3555.000000,614.000000,1.000000,1.000000,11.500000


## 7. Feature Scaling (Optional)

In [7]:
# TODO: Apply feature scaling to numeric features if needed
# Note: Tree-based models (like Random Forest) don't require scaling
# But it's useful for other algorithms

print("Applying feature scaling to numeric features...")

# Identify numeric columns to scale (exclude target and ID columns)
exclude_cols = ['effectiveness', 'effectiveness_encoded', 'rating', 'drugName', 
                'condition', 'drugName_top', 'condition_top', 'review']
numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns
cols_to_scale = [col for col in numeric_cols if col not in exclude_cols and not col.startswith('drug_') 
                 and not col.startswith('condition_')]

if len(cols_to_scale) > 0:
    # Create a copy for scaled features
    scaler = StandardScaler()
    
    # Scale the features
    df_encoded[cols_to_scale] = scaler.fit_transform(df_encoded[cols_to_scale])
    
    print(f"Scaled {len(cols_to_scale)} numeric features:")
    print(cols_to_scale)
    print("\nScaled features summary:")
    display(df_encoded[cols_to_scale].describe())
else:
    print("No numeric features to scale.")

Applying feature scaling to numeric features...
Scaled 8 numeric features:
['uniqueID', 'usefulCount', 'year', 'review_length', 'review_word_count', 'has_positive_keywords', 'has_negative_keywords', 'avg_word_length']

Scaled features summary:


,uniqueID,usefulCount,year,review_length,review_word_count,has_positive_keywords,has_negative_keywords,avg_word_length
count,2.473000e+03,2.473000e+03,2.473000e+03,2.473000e+03,2.473000e+03,2.473000e+03,2.473000e+03,2.473000e+03
mean,3.663332e-17,2.011241e-17,5.447590e-15,-8.906925e-17,6.464703e-17,1.246251e-16,8.116794e-17,-2.815737e-16
std,1.000202e+00,1.000202e+00,1.000202e+00,1.000202e+00,1.000202e+00,1.000202e+00,1.000202e+00,1.000202e+00
min,-2.017381e+00,-1.036087e+00,-1.986798e+00,-1.397833e+00,-1.384227e+00,-1.027470e+00,-1.395549e+00,-1.883506e+00
25%,-8.576180e-01,-6.885557e-01,-8.870401e-01,-8.049972e-01,-8.140595e-01,-1.027470e+00,-1.395549e+00,-5.656169e-01
50%,-1.536835e-01,-2.715184e-01,2.127177e-01,-1.911393e-01,-1.982789e-01,9.732647e-01,7.165639e-01,-1.701913e-01
75%,8.281903e-01,3.887908e-01,9.458896e-01,7.002161e-01,6.911819e-01,9.732647e-01,7.165639e-01,3.150818e-01
max,1.811663e+00,6.470585e+00,1.312476e+00,1.353237e+01,1.259627e+01,9.732647e-01,7.165639e-01,9.759244e+00


## 8. Save Processed Data

In [8]:
# TODO: Save ML-ready dataset
output_path = '../data/processed/pain_meds_ml_ready.csv'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save to CSV
df_encoded.to_csv(output_path, index=False)

print(f"Processed data saved to: {output_path}")
print("\n" + "=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"Original features: {df.shape[1]}")
print(f"Final features: {df_encoded.shape[1]}")
print(f"Total records: {len(df_encoded):,}")
print(f"\nTarget variable: effectiveness_encoded")
print(f"Feature categories created:")
print(f"  - Drug encoding: {drug_dummies.shape[1]} features")
print(f"  - Condition encoding: {condition_dummies.shape[1]} features")
if 'review' in df.columns:
    print(f"  - Text features: {len(text_features)} features")
print("=" * 60)

# Display sample of final dataset
print("\nSample of processed data:")
display(df_encoded.head())

Processed data saved to: ../data/processed/pain_meds_ml_ready.csv

FEATURE ENGINEERING SUMMARY
Original features: 12
Final features: 61
Total records: 2,473

Target variable: effectiveness_encoded
Feature categories created:
  - Drug encoding: 31 features
  - Condition encoding: 13 features
  - Text features: 5 features

Sample of processed data:


,uniqueID,drugName,condition,review,rating,date,usefulCount,year,effectiveness,effectiveness_encoded,drugName_top,condition_top,drug_Acetaminophen,drug_Acetaminophen / aspirin / caffeine,drug_Acetaminophen / butalbital,drug_Acetaminophen / butalbital / caffeine,drug_Acetaminophen / butalbital / caffeine / codeine,drug_Acetaminophen / caffeine / isometheptene mucate,drug_Acetaminophen / dichloralphenazone / isometheptene mucate,drug_Acetaminophen / diphenhydramine,drug_Acetaminophen / hydrocodone,drug_Acetaminophen / oxycodone,drug_Acetaminophen / pamabrom,drug_Acetaminophen / phenyltoloxamine,drug_Advil,drug_Aleve,drug_Aspirin,drug_Aspirin / butalbital / caffeine,drug_Aspirin / butalbital / caffeine / codeine,drug_Celebrex,drug_Diclofenac,drug_Diclofenac / misoprostol,drug_Esomeprazole / naproxen,drug_Famotidine / ibuprofen,drug_Ibuprofen,drug_Indomethacin,drug_Meloxicam,drug_Motrin,drug_Naproxen,drug_Naproxen / sumatriptan,drug_Other,drug_Oxycodone,drug_Tramadol,condition_back pain,condition_chronic pain,condition_cluster headaches,condition_headache,condition_juvenile rheumatoid arthritis,condition_migraine,condition_muscle pain,condition_neck pain,condition_osteoarthritis,condition_rheumatoid arthritis,condition_sciatica,condition_spondyloarthritis,condition_toothache,review_length,review_word_count,has_positive_keywords,has_negative_keywords,avg_word_length
0,1.117185,Oxycodone,chronic pain,"""I&#039;ve been taking oxycodone for roughly 5 years....I&#039;ve gone a few weekends without it...",8,18-Dec-16,-0.202012,0.945890,Effective,2,Oxycodone,chronic pain,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,1.923728,1.763096,0.973265,0.716564,0.140318
1,-0.712322,Aleve,back pain,"""I love Aleve! It makes all my lower back pain disappear, I feel like a new person.""",10,12-Aug-10,0.388791,-1.253626,Effective,2,Aleve,back pain,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,-1.061472,-1.019320,-1.027470,0.716564,-0.846500
2,0.828476,Meloxicam,osteoarthritis,"""I have been using Mobic to relieve the pain from my Spinal Fusion I had in March/2001. I was pr...",10,25-Sep-09,-0.132506,-1.620212,Effective,2,Meloxicam,osteoarthritis,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,-0.586363,-0.563186,-1.027470,0.716564,-0.349654
3,-1.376512,Acetaminophen / oxycodone,chronic pain,"""This med helps to take the edge off enough for the pain to be tolerable. I take HCL 15 mg one e...",6,10-Nov-17,-1.036087,1.312476,Partially Effective,1,Acetaminophen / oxycodone,chronic pain,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,-0.523295,-0.449152,0.973265,0.716564,-0.745493
4,-0.154256,Acetaminophen / butalbital / caffeine,headache,"""I have been suffering from terrible allergies due to hay fever. The allergies caused horrible ...",9,19-Sep-11,-1.036087,-0.887040,Effective,2,Acetaminophen / butalbital / caffeine,headache,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,0.035904,-0.152666,0.973265,